# 05 — Mlp Stacking / 堆叠方法

**Chapter 25 — File 5 of 5 / 第25章 — 第5个文件（共5个）**

---

## Summary / 总结

This script demonstrates **stacked generalization with neural net meta model on blobs dataset**.

本脚本演示 **stacked generalization with neural net meta model on blobs dataset**。

---
## Step 1 — stacked generalization with neural net meta model on blobs dataset

In [ ]:
from sklearn.datasets import make_blobs
from sklearn.metrics import accuracy_score
from keras.models import load_model
from keras.utils import to_categorical
from keras.utils import plot_model
from keras.models import Model
from keras.layers import Dense
from keras.layers.merge import concatenate
from numpy import argmax

---
## Step 2 — load models from file

In [ ]:
def load_all_models(n_models):
	all_models = list()
	for i in range(n_models):

---
## Step 3 — define filename for this ensemble

In [ ]:
filename = 'models/model_' + str(i + 1) + '.h5'

---
## Step 4 — load model from file

In [ ]:
model = load_model(filename)

---
## Step 5 — add to list of members

In [ ]:
all_models.append(model)
		print('>loaded %s' % filename)
	return all_models

---
## Step 6 — define stacked model from multiple member input models

In [ ]:
def define_stacked_model(members):

---
## Step 7 — update all layers in all models to not be trainable

In [ ]:
for i in range(len(members)):
		model = members[i]
		for layer in model.layers:

---
## Step 8 — make not trainable

In [ ]:
layer.trainable = False

---
## Step 9 — rename to avoid 'unique layer name' issue

In [ ]:
layer._name = 'ensemble_' + str(i+1) + '_' + layer.name

---
## Step 10 — define multi-headed input

In [ ]:
ensemble_visible = [model.input for model in members]

---
## Step 11 — concatenate merge output from each model

In [ ]:
ensemble_outputs = [model.output for model in members]
	merge = concatenate(ensemble_outputs)
	hidden = Dense(10, activation='relu')(merge)
	output = Dense(3, activation='softmax')(hidden)
	model = Model(inputs=ensemble_visible, outputs=output)

---
## Step 12 — plot graph of ensemble

In [ ]:
plot_model(model, show_shapes=True, to_file='model_graph.png')

---
## Step 13 — compile

In [ ]:
model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
	return model

---
## Step 14 — fit a stacked model

In [ ]:
def fit_stacked_model(model, inputX, inputy):

---
## Step 15 — prepare input data

In [ ]:
X = [inputX for _ in range(len(model.input))]

---
## Step 16 — encode output data

In [ ]:
inputy_enc = to_categorical(inputy)

---
## Step 17 — fit model

In [ ]:
model.fit(X, inputy_enc, epochs=300, verbose=0)

---
## Step 18 — make a prediction with a stacked model

In [ ]:
def predict_stacked_model(model, inputX):

---
## Step 19 — prepare input data

In [ ]:
X = [inputX for _ in range(len(model.input))]

---
## Step 20 — make prediction

In [ ]:
return model.predict(X, verbose=0)

---
## Step 21 — generate 2d classification dataset

In [ ]:
X, y = make_blobs(n_samples=1100, centers=3, n_features=2, cluster_std=2, random_state=2)

---
## Step 22 — split into train and test

In [ ]:
n_train = 100
trainX, testX = X[:n_train, :], X[n_train:, :]
trainy, testy = y[:n_train], y[n_train:]

---
## Step 23 — load all models

In [ ]:
n_members = 5
members = load_all_models(n_members)
print('Loaded %d models' % len(members))

---
## Step 24 — define ensemble model

In [ ]:
stacked_model = define_stacked_model(members)

---
## Step 25 — fit stacked model on test dataset

In [ ]:
fit_stacked_model(stacked_model, testX, testy)

---
## Step 26 — make predictions and evaluate

In [ ]:
yhat = predict_stacked_model(stacked_model, testX)
yhat = argmax(yhat, axis=1)
acc = accuracy_score(testy, yhat)
print('Stacked Test Accuracy: %.3f' % acc)

---
## Learning Notes / 学习笔记

- **概念**: stacked generalization with neural net meta model on blobs dataset 是机器学习中的常用技术。  
  *stacked generalization with neural net meta model on blobs dataset is a common technique in machine learning.*

- **ML 应用**: 本示例展示了如何在实践中应用该技术。  
  *This example shows how to apply the technique in practice.*

---
## Complete Code / 完整代码一览

Below is the full code for quick reference. / 以下是完整代码，供快速参考。

In [ ]:
# ===============================
# Mlp Stacking / 堆叠方法
# Complete Code / 完整代码
# ===============================

# stacked generalization with neural net meta model on blobs dataset
from sklearn.datasets import make_blobs
from sklearn.metrics import accuracy_score
from keras.models import load_model
from keras.utils import to_categorical
from keras.utils import plot_model
from keras.models import Model
from keras.layers import Dense
from keras.layers.merge import concatenate
from numpy import argmax

# load models from file
def load_all_models(n_models):
	all_models = list()
	for i in range(n_models):
		# define filename for this ensemble
		filename = 'models/model_' + str(i + 1) + '.h5'
		# load model from file
		model = load_model(filename)
		# add to list of members
		all_models.append(model)
		print('>loaded %s' % filename)
	return all_models

# define stacked model from multiple member input models
def define_stacked_model(members):
	# update all layers in all models to not be trainable
	for i in range(len(members)):
		model = members[i]
		for layer in model.layers:
			# make not trainable
			layer.trainable = False
			# rename to avoid 'unique layer name' issue
			layer._name = 'ensemble_' + str(i+1) + '_' + layer.name
	# define multi-headed input
	ensemble_visible = [model.input for model in members]
	# concatenate merge output from each model
	ensemble_outputs = [model.output for model in members]
	merge = concatenate(ensemble_outputs)
	hidden = Dense(10, activation='relu')(merge)
	output = Dense(3, activation='softmax')(hidden)
	model = Model(inputs=ensemble_visible, outputs=output)
	# plot graph of ensemble
	plot_model(model, show_shapes=True, to_file='model_graph.png')
	# compile
	model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
	return model

# fit a stacked model
def fit_stacked_model(model, inputX, inputy):
	# prepare input data
	X = [inputX for _ in range(len(model.input))]
	# encode output data
	inputy_enc = to_categorical(inputy)
	# fit model
	model.fit(X, inputy_enc, epochs=300, verbose=0)

# make a prediction with a stacked model
def predict_stacked_model(model, inputX):
	# prepare input data
	X = [inputX for _ in range(len(model.input))]
	# make prediction
	return model.predict(X, verbose=0)

# generate 2d classification dataset
X, y = make_blobs(n_samples=1100, centers=3, n_features=2, cluster_std=2, random_state=2)
# split into train and test
n_train = 100
trainX, testX = X[:n_train, :], X[n_train:, :]
trainy, testy = y[:n_train], y[n_train:]
# load all models
n_members = 5
members = load_all_models(n_members)
print('Loaded %d models' % len(members))
# define ensemble model
stacked_model = define_stacked_model(members)
# fit stacked model on test dataset
fit_stacked_model(stacked_model, testX, testy)
# make predictions and evaluate
yhat = predict_stacked_model(stacked_model, testX)
yhat = argmax(yhat, axis=1)
acc = accuracy_score(testy, yhat)
print('Stacked Test Accuracy: %.3f' % acc)